### <span style="color:blue">Define paths</span>

In [ ]:
# Check paths
import os
print(os.getcwd())

print(os.path.exists("../data/documents.csv"))

In [ ]:
# Define paths

from pathlib import Path

BASE_DIR = Path().resolve().parent  # project
doc_csv_path = BASE_DIR / "data" / "documents.csv"
doc_jsonl_path = BASE_DIR / "data" / "documents.jsonl"

queries_csv_path = BASE_DIR / "data" / "queries.csv"
queries_jsonl_path = BASE_DIR / "data" / "queries.jsonl"


### <span style="color:blue">Preprocess</span>

In [ ]:
# Function to convert csv file to jsonl

import csv
import json

def csv_to_jsonl(csv_path, jsonl_path, encoding="utf-8"):
    with open(csv_path, mode="r", encoding=encoding, newline="") as csv_file:
        reader = csv.DictReader(csv_file)

        with open(jsonl_path, mode="w", encoding=encoding) as jsonl_file:
            for row in reader:
                json_line = json.dumps(row, ensure_ascii=False)
                jsonl_file.write(json_line + "\n")



In [ ]:
# Function reading jsonl file

def read_json_data(jsonl_path):
    all_json_data = [] # a list of json records
    with open(jsonl_path, "r", encoding="utf-8") as file:
        for line in file:
            json_data = json.loads(line)
            all_json_data.append(json_data)
    return all_json_data

In [ ]:
# Function get_query that extracts all the text for a specific id

def get_query(all_json_queries, search_id):
    for q in all_json_queries:
        if q['ID'] == search_id:
            text = q['Text']
            return text

In [ ]:
# Conversion of the file documents.csv to documents.jsonl

csv_to_jsonl(doc_csv_path, doc_jsonl_path)

In [ ]:
# Conversion of the file queries.csv to queries.jsonl

csv_to_jsonl(queries_csv_path, queries_jsonl_path)

In [ ]:
# List of data in documents.jsonl

all_json_data = read_json_data(doc_jsonl_path)

# print 5 first data
# for item in all_json_data[:5]:
#     print(f"id: {item['ID']}\ntext: {item['Text']}\n")

In [ ]:
# List of queries in queries.jsonl

all_json_queries = read_json_data(queries_jsonl_path)

# print 5 first queries
# for item in all_json_queries[:5]:
#     print(f"id: {item['ID']}\ntext: {item['Text']}\n")

In [ ]:
from datasets import load_dataset
from cleantext import clean

https://pypi.org/project/clean-text/

from cleantext import clean

```clean("some input",
    fix_unicode=True,               # fix various unicode errors
    to_ascii=True,                  # transliterate to closest ASCII representation
    lower=True,                     # lowercase text
    no_line_breaks=False,           # fully strip line breaks as opposed to only normalizing them
    no_urls=False,                  # replace all URLs with a special token
    no_emails=False,                # replace all email addresses with a special token
    no_phone_numbers=False,         # replace all phone numbers with a special token
    no_numbers=False,               # replace all numbers with a special token
    no_digits=False,                # replace all digits with a special token
    no_currency_symbols=False,      # replace all currency symbols with a special token
    no_punct=False,                 # remove punctuations
    replace_with_punct="",          # instead of removing punctuations you may replace them
    replace_with_url="<URL>",
    replace_with_email="<EMAIL>",
    replace_with_phone_number="<PHONE>",
    replace_with_number="<NUMBER>",
    replace_with_digit="0",
    replace_with_currency_symbol="<CUR>",
    lang="en"                       # set to 'de' for German special handling
)```

https://huggingface.co/docs/datasets/about_map_batch

### example overwrites the existing "a" column with duplicated values

```dataset = Dataset.from_dict({"a": [0, 1, 2]})

duplicated_dataset = dataset.map(

    lambda batch: {"a": [x for x in batch["a"] for _ in range(2)]},

    batched=True
)```



In [ ]:
# Preprocessing function cleans the "Text"

def preprocess(batch):
    texts = batch["Text"] # batch στην dataset.map()
    cleaned = []

    for text in texts:
        # Basic cleaning χρησιμοποιεί τη συνάρτηση clean από clean-text package pypi.org
        t = clean(
            text,
            fix_unicode=True,           # διορθώνει περίεργους χαρακτήρες
            to_ascii=False,             # κρατάει non-ASCII (π.χ. ονόματα)
            lower=False,                # όχι lowercase για BERT-like μοντέλα
            no_urls=True,
            no_emails=True,
            no_phone_numbers=True,
            no_line_breaks=True,
            no_punct=False,             # κρατάμε σημεία στίξης (τα χρειάζεται ο tokenizer)
            no_digits=False,            
            no_currency_symbols=False
        )

        # Light normalization
        t = t.strip()                   # αφαίρεση περιττών κενών
        t = " ".join(t.split())         # normalize whitespace

        cleaned.append(t)

    batch["clean_text"] = cleaned
    return batch

In [ ]:
# Mapping for data

dataset = load_dataset("json", data_files=str(doc_jsonl_path))["train"] # dataset hugging face
dataset = dataset.map(preprocess, batched=True)

print(f"type: {type(dataset)}\n")

print(f"dataset length: {len(dataset)}\n")

print(f"{dataset}\n")

for d in dataset[:5]["clean_text"]:
    print(f"{d}\n")


In [ ]:
# Mapping for queries

queries = load_dataset("json", data_files=str(queries_jsonl_path))["train"]
queries = queries.map(preprocess, batched=True)

print(f"type: {type(queries)}\n")

print(f"queries length: {len(queries)}\n")

print(f"{queries}")

for q in queries[:5]["clean_text"]:
    print(f"{q}\n")

In [ ]:
# Cleaned data, queries

clean_dataset = dataset.remove_columns(
    [col for col in dataset.column_names if col not in ["ID", "clean_text"]]
)

clean_dataset = clean_dataset.rename_column(
    "clean_text", "text"
)

clean_queries = queries.remove_columns(
    [col for col in queries.column_names if col not in ["ID", "clean_text"]]
)

clean_queries = clean_queries.rename_column(
    "clean_text", "text"
)

print(f"clean_dataset:\n{clean_dataset}\n\nclean_queries:\n{clean_queries}")

### <span style="color:blue">Creating embeddings</span>
**Uncomment the code if you want to create embeddings**

In [ ]:
from sentence_transformers import SentenceTransformer

# model = SentenceTransformer("sentence-transformers/all-MiniLM-L6-v2")

# def embed(batch):
#     return {
#         "embeddings": model.encode(
#             batch["text"],
#             batch_size=32,
#             convert_to_numpy=True,
#             normalize_embeddings=True
#         )
#     }

# clean_dataset = clean_dataset.map(embed, batched=True)

# clean_queries = clean_queries.map(embed, batched=True)

### <span style="color:blue">With the following code, the embeddings are saved in the artifacts folder</span>

In [ ]:
# clean_dataset.save_to_disk("artifacts/clean_dataset_with_embeddings")

In [ ]:
# clean_queries.save_to_disk("artifacts/clean_queries_with_embeddings")

### <span style="color:blue">Import embeddings from the artifacts folder</span>

In [ ]:
# import embeddings from artifacts folder

from datasets import load_from_disk

clean_dataset = load_from_disk(
    "artifacts/clean_dataset_with_embeddings"
)
clean_queries = load_from_disk(
    "artifacts/clean_queries_with_embeddings"
)


In [ ]:
print(clean_dataset.features)

In [ ]:
print(clean_queries.features)

In [ ]:
print(clean_dataset["embeddings"])

In [ ]:
print(clean_queries["embeddings"])

### <span style="color:blue">Creating index with FAISS</span>

In [ ]:
import numpy as np
import faiss

# Converting embeddings to np.array
embeddings = np.array(clean_dataset["embeddings"]).astype("float32")

dim = 384

# Initialize
faiss_index = faiss.IndexFlatIP(dim) # cosine similarity

# Filling the index
faiss_index.add(embeddings)

### <span style="color:blue">Testing of the index</span>

In [ ]:
print(embeddings[0][:10])

columns = len(embeddings[0][:])
print(f"columns: {columns}")

size = len(embeddings[:])
print(f"size: {size}")


In [ ]:
print(f"index size: {faiss_index.ntotal}")

In [ ]:
print(f"Vector dimensions: {faiss_index.d}")

In [ ]:
print(f"Is trained:{faiss_index.is_trained}")

In [ ]:
# Testing/Debugging
print("Index type:", type(faiss_index))
print("Vectors:", faiss_index.ntotal)
print("Dim:", faiss_index.d)

query_embeddings = np.array(clean_queries["embeddings"]).astype("float32")

D, I = faiss_index.search(query_embeddings, 5)

for score, idx in zip(D[0], I[0]):
    print(f"idx: {idx} score: s{score} ID: {clean_dataset[idx]["ID"]}")


### <span style="color:blue">To understand how the index works</span>

In [ ]:
# k nearest neighbors for a query (Which rows of the dataset/embeddings are the top-k)
   
k = 2

score, indices = faiss_index.search(query_embeddings, k)

# For k=2 nearest neighbors 
# The 1st neighbor with row 209 in the index has similarity score: 0.99689716
# The 2nd neighbor with row 13343 in the index has similarity score: 0.6836617 (less relevant neighbor) 
print(f"score: {score[0]}\n\nindices: {indices[0]}")

In [ ]:
# k=2 nearest neighbors for a query and the related id and text from the dataset

for s, idx in zip(score[0], indices[0]):
    doc = clean_dataset[idx]
    print(f"doc_id={doc['ID']}, score={s:.4f}, text={doc['text'][:50]}...")

### <span style="color:blue">Export run files for trec_eval</span>

In [ ]:
import numpy as np

# query embeddings (shape = (num_queries, dim))
query_embeddings = np.array(clean_queries["embeddings"]).astype("float32")

search_ids = ["Q01", "Q02", "Q03", "Q04", "Q05", "Q06", "Q07", "Q08", "Q09", "Q10"]

ks = [20, 30, 50]

run_name = "faiss_minilm"

for k in ks:
    # Batch search για όλα τα queries
    D, I = faiss_index.search(query_embeddings, k) # D: score, I: indicies of neighbors

    # trec_eval run file
    filename = f"faiss_run_k{k}.txt"
    with open(filename, "w") as f:
        for q_idx, qid in enumerate(search_ids):
            for rank, (score, idx) in enumerate(zip(D[q_idx], I[q_idx]), start=1):
                doc = clean_dataset[idx]
                f.write(f"{qid} Q0 {doc['ID']} {rank} {score} {run_name}\n")

    print(f"Run file for k={k} written to {filename}")
